# Date Cleaning Note

There are 4 new tables created after cleaning process

- **reviews_latest** where we keep only one review per order using the latest one
- **orders_labeled** where every columns and add with additional calculated columns and an exclusion_reason case
- **orders_clean** where we only drop all invalid order and kept only order that is delivered and has review while inside the analysis period.
- **items_clean** where we keep only one row per item for the clean orders, with the english category.

**Setup**

Loads the SQL extension and connects to the Olist database.

In [6]:
%load_ext sql
%sql duckdb:///../data/olist.duckdb
%config SqlMagic.displaylimit = 20

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


**One review per order**

Create a reviews_lastest tables since some order have more than one review. Keep only the row that have the latest review.

In [7]:
%%sql

CREATE OR REPLACE TABLE reviews_latest AS
SELECT
    order_id,
    review_id,
    review_score,
    review_creation_date
FROM order_reviews
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_id
    ORDER BY review_answer_timestamp DESC, review_creation_date DESC
) = 1

Running query in 'duckdb:///../data/olist.duckdb'

Count


**Label every order**

Builds one row per order with everything needed for analysis:

- items_per_order first adds up the items, revenue and shipping cost for each order, so the join does not duplicate orders.
- Customer details come from the customers table, using the unique customer ID for the real person.
- delivery_days is the number of days from purchase to delivery.
- days_late is the number of days from the estimated date to the actual delivery. Positive means late, negative means early.
- is_late is true when the order arrived after the estimated date.
- exclusion_reason gives each order one label. The rules are checked in order and the first match wins. Orders that pass every rule are labelled Kept.

In [8]:
%%sql

CREATE OR REPLACE TABLE orders_labeled AS
WITH items_per_order AS (
    SELECT
        order_id,
        COUNT(*) AS n_items,
        SUM(price) AS revenue,
        SUM(freight_value) AS freight
    FROM order_items
    GROUP BY order_id
)
SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    DATE_DIFF('day', o.order_purchase_timestamp::DATE, o.order_delivered_customer_date::DATE) AS delivery_days,
    DATE_DIFF('day', o.order_estimated_delivery_date::DATE, o.order_delivered_customer_date::DATE) AS days_late,
    DATE_DIFF('day', o.order_estimated_delivery_date::DATE, o.order_delivered_customer_date::DATE) > 0 AS is_late,
    r.review_score,
    i.n_items,
    i.revenue,
    i.freight,
    CASE
        WHEN o.order_status <> 'delivered'                                THEN '1. Not delivered'
        WHEN o.order_delivered_customer_date IS NULL                      THEN '2. No delivery date'
        WHEN o.order_delivered_customer_date < o.order_purchase_timestamp THEN '3. Impossible dates'
        WHEN o.order_purchase_timestamp <  '2017-01-01'
          OR o.order_purchase_timestamp >= '2018-09-01'                   THEN '4. Outside analysis period'
        WHEN r.review_score IS NULL                                       THEN '5. No review'
        ELSE 'Kept'
    END AS exclusion_reason
FROM orders AS o
JOIN customers AS c            ON o.customer_id = c.customer_id
LEFT JOIN reviews_latest AS r  ON o.order_id = r.order_id
LEFT JOIN items_per_order AS i ON o.order_id = i.order_id


Running query in 'duckdb:///../data/olist.duckdb'

Count


**Spot check: orders with 20 items**

In [9]:
%%sql

SELECT *
from orders_labeled
order by customer_unique_id, order_id

Running query in 'duckdb:///../data/olist.duckdb'

order_id,customer_unique_id,customer_state,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,days_late,is_late,review_score,n_items,revenue,freight,exclusion_reason
e22acc9c116caa3f2b7121bbb380d08e,0000366f3b9a7992bf8c76cfdf3221e2,SP,2018-05-10 10:56:27,2018-05-16 20:48:37,2018-05-21 00:00:00,6,-5,False,5,1,129.9,12.0,Kept
3594e05a005ac4d06a72673270ef9ec9,0000b849f77a49e4a4ce2b2a4ca5be3f,SP,2018-05-07 11:11:27,2018-05-10 18:02:42,2018-05-15 00:00:00,3,-5,False,4,1,18.9,8.29,Kept
b33ec3b699337181488304f362a6b734,0000f46a3911fa3c0805444483337064,SC,2017-03-10 21:05:03,2017-04-05 14:38:47,2017-04-07 00:00:00,26,-2,False,3,1,69.0,17.22,Kept
41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,PA,2017-10-12 20:29:41,2017-11-01 21:23:05,2017-11-13 00:00:00,20,-12,False,4,1,25.99,17.63,Kept
d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,SP,2017-11-14 19:45:42,2017-11-27 23:08:56,2017-12-05 00:00:00,13,-8,False,5,1,180.0,16.89,Kept
3e470077b690ea3e3d501cffb5e0c499,0004bd2a26a76fe21f786e4fbd80607f,SP,2018-04-05 19:33:16,2018-04-07 16:12:43,2018-04-19 00:00:00,2,-12,False,4,1,154.0,12.98,Kept
d0028facea13f508e880202d7097a5a1,00050ab1314c0e55a6ca13cf7181fecf,SP,2018-04-20 12:57:23,2018-04-27 12:08:59,2018-05-09 00:00:00,7,-12,False,4,1,27.99,7.39,Kept
44e608f2db00c74a1fe329de44416a4e,00053a61a98854899e70ed204dd4bafe,PR,2018-02-28 11:15:41,2018-03-16 19:02:51,2018-03-26 00:00:00,16,-10,False,1,2,382.0,37.18,Kept
ae76bef74b97bcb0b3e355e60d9a6f9c,0005e1862207bf6ccc02e4228effd9a0,RJ,2017-03-04 23:32:12,2017-03-09 08:33:08,2017-04-06 00:00:00,5,-28,False,4,1,135.0,15.12,Kept
01b330808c5819a6a3cb79b72f0b8288,0005ef4cd20d2893f0d9fbd94d3c0d97,MA,2018-03-12 15:22:12,2018-05-05 12:38:38,2018-04-04 00:00:00,54,31,True,1,1,104.9,24.86,Kept


**How many orders each rule removes**

In [10]:
%%sql
select exclusion_reason, count(*) as n_orders
from orders_labeled
group by exclusion_reason

Running query in 'duckdb:///../data/olist.duckdb'

exclusion_reason,n_orders
2. No delivery date,8
5. No review,643
4. Outside analysis period,267
1. Not delivered,2963
Kept,95560


**Clean orders table**

Keeps only the orders labelled Kept and saves them as orders_clean. The label column is dropped because every row would have the same value. This is the main table for analysis.

In [11]:
%%sql
CREATE OR REPLACE TABLE orders_clean AS
SELECT * exclude (exclusion_reason)
FROM orders_labeled
WHERE exclusion_reason = 'Kept'

Running query in 'duckdb:///../data/olist.duckdb'

Count


**Clean items table**

Keeps one row per item, but only for orders in orders_clean. It adds the product category in English. If there is no English name, it uses the Portuguese name, and if there is no category at all, it uses Unknown.

In [12]:
%%sql
CREATE OR REPLACE TABLE items_clean AS
SELECT
    i.order_id,
    i.order_item_id,
    i.product_id,
    i.seller_id,
    coalesce(pc.product_category_name_english, p.product_category_name, 'Unknown') AS category,
    i.price,
    i.freight_value
FROM order_items i
JOIN orders_clean oc ON i.order_id = oc.order_id
LEFT JOIN products AS p ON i.product_id = p.product_id
LEFT JOIN product_category_name_translation AS pc ON p.product_category_name = pc.product_category_name

Running query in 'duckdb:///../data/olist.duckdb'

Count


**Spot check: orders with many items**

In [17]:
%%sql
SELECT *
FROM items_clean
LIMIT 5

Running query in 'duckdb:///../data/olist.duckdb'

order_id,order_item_id,product_id,seller_id,category,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,199.9,18.14


**Final check on orders_clean**

Confirms the clean table is ready. There are 95,560 rows and 95,560 orders, so there is one row per order. No reviews, delivery times or items are missing, and no delivery time is negative. Purchases run from 5 January 2017 to 29 August 2018. 6.7 percent of the kept orders arrived late.

In [14]:
%%sql

SELECT
    COUNT(*)                                      AS rows,
    COUNT(DISTINCT order_id)                      AS orders,
    COUNT(*) FILTER (WHERE review_score IS NULL)  AS missing_review,
    COUNT(*) FILTER (WHERE delivery_days IS NULL) AS missing_delivery,
    COUNT(*) FILTER (WHERE n_items IS NULL)       AS missing_items,
    COUNT(*) FILTER (WHERE delivery_days < 0)     AS negative_delivery,
    MIN(order_purchase_timestamp)                 AS first_purchase,
    MAX(order_purchase_timestamp)                 AS last_purchase,
    ROUND(100 * AVG(is_late::INT), 1)             AS late_pct
FROM orders_clean

Running query in 'duckdb:///../data/olist.duckdb'

rows,orders,missing_review,missing_delivery,missing_items,negative_delivery,first_purchase,last_purchase,late_pct
95560,95560,0,0,0,0,2017-01-05 11:56:06,2018-08-29 15:00:37,6.7
